In [ ]:
# Fine-tunes VideoMAE V2 on the RHAD dataset and exports its validation and test predictions.

In [ ]:
# VIDEOMAE — CELLULE 1
# Vérification des inputs, du dataset et des GPU

from pathlib import Path
import torch

DATASET_ROOT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "training/Training"
)

OLD_CHECKPOINT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "checkpoint39/checkpoint-39.pth"
)

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mkv", ".mov", ".webm"
}

print("=" * 70)
print("GPU")
print("=" * 70)

print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPU :", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index} :",
        torch.cuda.get_device_name(index)
    )

print("\n" + "=" * 70)
print("DATASET RHAD")
print("=" * 70)

assert DATASET_ROOT.exists(), (
    f"Dataset introuvable : {DATASET_ROOT}"
)

classes = sorted([
    folder.name
    for folder in DATASET_ROOT.iterdir()
    if folder.is_dir()
])

videos = [
    path
    for path in DATASET_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in VIDEO_EXTENSIONS
    )
]

print("Chemin :", DATASET_ROOT)
print("Nombre de classes :", len(classes))
print("Nombre de vidéos :", len(videos))
print("Classes :", classes)

assert len(classes) == 12, (
    f"12 classes attendues, {len(classes)} trouvées."
)

assert len(videos) == 3248, (
    f"3248 vidéos attendues, {len(videos)} trouvées."
)

print("\n" + "=" * 70)
print("ANCIEN CHECKPOINT")
print("=" * 70)

print("Présent :", OLD_CHECKPOINT.exists())
print("Chemin :", OLD_CHECKPOINT)
print(
    "Statut : IGNORÉ — il ne sera ni copié ni chargé."
)

print("\nVÉRIFICATION DE DÉPART VALIDÉE")

In [ ]:
# VIDEOMAE — CELLULE 2
# Création d'un environnement Python 3.10 isolé

from pathlib import Path
import shutil
import subprocess
import sys

ENV_DIR = Path("/kaggle/working/vmae_env")

# Installer uv, utilisé pour créer rapidement l'environnement
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "uv",
    ],
    check=True,
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv n'a pas été trouvée."
    )

print("uv trouvé :", UV)

# Supprimer uniquement un éventuel ancien environnement VideoMAE
shutil.rmtree(
    ENV_DIR,
    ignore_errors=True,
)

# Installer Python 3.10
subprocess.run(
    [
        UV,
        "python",
        "install",
        "3.10",
    ],
    check=True,
)

# Créer l'environnement isolé
subprocess.run(
    [
        UV,
        "venv",
        "--python",
        "3.10",
        str(ENV_DIR),
    ],
    check=True,
)

PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python absent : {PY}"
)

# Vérification
subprocess.run(
    [
        PY,
        "--version",
    ],
    check=True,
)

print("Environnement créé :", ENV_DIR)
print("Python utilisé :", PY)
print("ENVIRONNEMENT VIDEOMAE VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 3
# Installation de PyTorch et des dépendances VideoMAEv2

from pathlib import Path
import shutil
import subprocess

ENV_DIR = Path("/kaggle/working/vmae_env")
PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python de l'environnement absent : {PY}"
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv est introuvable."
    )

print("=" * 70)
print("1. INSTALLATION DE PYTORCH + CUDA 11.8")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
        "torch==2.1.2",
        "torchvision==0.16.2",
        "torchaudio==2.1.2",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("2. INSTALLATION DES DÉPENDANCES VIDEOMAE")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,

        # NumPy 1.x pour éviter les incompatibilités
        # avec les anciennes bibliothèques VideoMAE
        "numpy==1.26.4",

        # Version recommandée par VideoMAEv2
        "timm==0.4.12",

        # Lecture et traitement vidéo
        "decord==0.6.0",
        "av",
        "opencv-python-headless",
        "Pillow",

        # Calcul scientifique
        "einops",
        "scipy",
        "pandas",

        # Logs et résultats
        "tensorboardX",
        "matplotlib",
        "scikit-learn",

        # Téléchargement du modèle préentraîné
        "huggingface_hub",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("3. VÉRIFICATION DES INSTALLATIONS")
print("=" * 70)

verification_code = r"""
import sys

import torch
import torchvision
import timm
import numpy
import decord
import av
import cv2
import einops
import scipy
import pandas
import tensorboardX
import sklearn
import huggingface_hub

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("CUDA dispo   :", torch.cuda.is_available())
print("Nombre GPU   :", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}       :",
        torch.cuda.get_device_name(index)
    )

print("timm          :", timm.__version__)
print("NumPy         :", numpy.__version__)
print("decord        :", decord.__version__)
print("PyAV          :", av.__version__)
print("OpenCV        :", cv2.__version__)
print("SciPy         :", scipy.__version__)
print("Pandas        :", pandas.__version__)

# Petit calcul GPU réel
if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible dans l'environnement.")

x = torch.randn(
    128,
    128,
    device="cuda",
)

y = x @ x.T

assert y.shape == (128, 128)
assert torch.isfinite(y).all()

print("Test calcul GPU : VALIDÉ")
print("DÉPENDANCES VIDEOMAE VALIDÉES")
"""

subprocess.run(
    [
        PY,
        "-c",
        verification_code,
    ],
    check=True,
)

In [ ]:
# VIDEOMAE — CELLULE 4A
# Télécharger exactement la version utilisée du dépôt

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
COMMIT = "29eab1e8a588d1b3ec0cdec7b03a86cca491b74b"

shutil.rmtree(REPO_DIR, ignore_errors=True)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/OpenGVLab/VideoMAEv2.git",
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run(
    ["git", "checkout", COMMIT],
    cwd=str(REPO_DIR),
    check=True,
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=str(REPO_DIR),
    text=True,
).strip()

print("Commit téléchargé :", actual_commit)
assert actual_commit == COMMIT
print("DÉPÔT VIDEOMAEV2 VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 4B
# Vérification du dépôt et création du modèle

from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

assert REPO_DIR.exists(), (
    f"Dépôt absent : {REPO_DIR}"
)

assert Path(PY).exists(), (
    f"Environnement Python absent : {PY}"
)

# ============================================================
# 1. Vérification des vrais chemins du dépôt
# ============================================================

required_files = [
    REPO_DIR / "run_class_finetuning.py",
    REPO_DIR / "dataset" / "build.py",
    REPO_DIR / "dataset" / "datasets.py",
    REPO_DIR / "models" / "__init__.py",
    REPO_DIR / "models" / "modeling_finetune.py",
    REPO_DIR / "utils.py",
]

print("=" * 70)
print("1. FICHIERS ESSENTIELS")
print("=" * 70)

for file_path in required_files:
    exists = file_path.exists()

    print(
        file_path.relative_to(REPO_DIR),
        "->",
        exists,
    )

    assert exists, (
        f"Fichier essentiel absent : {file_path}"
    )

# ============================================================
# 2. Diagnostic du script d'entraînement
# ============================================================

training_script = (
    REPO_DIR / "run_class_finetuning.py"
).read_text(
    encoding="utf-8"
)

dataset_build = (
    REPO_DIR / "dataset" / "build.py"
).read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("2. DIAGNOSTIC DU SCRIPT")
print("=" * 70)

print(
    "Import obligatoire de DeepSpeed :",
    "import deepspeed" in training_script,
)

print(
    "Option --finetune présente :",
    "--finetune" in training_script,
)

print(
    "Option --resume présente :",
    "--resume" in training_script,
)

print(
    "Option --save_ckpt présente :",
    "--save_ckpt" in training_script,
)

print(
    "Dataset custom déjà supporté :",
    "'custom'" in dataset_build
    or '"custom"' in dataset_build,
)

# ============================================================
# 3. Test de création de VideoMAE
# ============================================================

print("\n" + "=" * 70)
print("3. TEST DE CRÉATION DU MODÈLE")
print("=" * 70)

test_code = r"""
import torch
import timm

# Cet import enregistre les modèles VideoMAE dans timm
import models

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_path_rate=0.1,
    use_mean_pooling=True,
    init_scale=0.001,
)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

classifier = model.get_classifier()

print("Modèle créé :", model.__class__.__name__)
print("Nombre de paramètres :", f"{number_of_parameters:,}")
print(
    "Nombre de classes :",
    getattr(classifier, "out_features", "non détecté"),
)
print("Frames :", 16)
print("Tubelet size :", 2)

assert getattr(classifier, "out_features", None) == 12

print("IMPORT ET CRÉATION VIDEOMAE VALIDÉS")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("Messages système / erreurs :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test de création VideoMAE a échoué."
    )

print("\nDÉPÔT VIDEOMAEV2 VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 5
# Adaptation du dépôt à RHAD, sans DeepSpeed

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

RUN_FILE = REPO_DIR / "run_class_finetuning.py"
BUILD_FILE = REPO_DIR / "dataset" / "build.py"

assert RUN_FILE.exists(), RUN_FILE
assert BUILD_FILE.exists(), BUILD_FILE
assert Path(PY).exists(), PY

# ============================================================
# 1. SAUVEGARDES DU CODE ORIGINAL
# ============================================================

for file_path in [RUN_FILE, BUILD_FILE]:
    backup_path = file_path.with_suffix(
        file_path.suffix + ".original"
    )

    if not backup_path.exists():
        shutil.copy2(
            file_path,
            backup_path,
        )

        print(
            "Sauvegarde créée :",
            backup_path,
        )
    else:
        print(
            "Sauvegarde déjà présente :",
            backup_path,
        )

# ============================================================
# 2. RENDRE DEEPSPEED OPTIONNEL
# ============================================================

run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

if (
    "try:\n    import deepspeed"
    not in run_text
):
    assert "import deepspeed" in run_text

    run_text = run_text.replace(
        "import deepspeed",
        """try:
    import deepspeed
except ImportError:
    deepspeed = None""",
        1,
    )

# Sécuriser l'option --enable_deepspeed
old_deepspeed_block = """    if known_args.enable_deepspeed:
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

new_deepspeed_block = """    if known_args.enable_deepspeed:
        if deepspeed is None:
            raise ImportError(
                "DeepSpeed a été demandé mais il n'est pas installé."
            )
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

if old_deepspeed_block in run_text:
    run_text = run_text.replace(
        old_deepspeed_block,
        new_deepspeed_block,
        1,
    )

# ============================================================
# 3. AJOUTER RHAD AUX DATASETS AUTORISÉS
# ============================================================

data_argument_start = run_text.find(
    "parser.add_argument(\n"
    "        '--data_set'"
)

assert data_argument_start != -1, (
    "Argument --data_set introuvable."
)

data_argument_end = run_text.find(
    "type=str",
    data_argument_start,
)

assert data_argument_end != -1

data_argument_block = run_text[
    data_argument_start:data_argument_end
]

if "'RHAD'" not in data_argument_block:
    assert "'MIT'" in data_argument_block

    patched_argument_block = (
        data_argument_block.replace(
            "'MIT'",
            "'MIT', 'RHAD'",
            1,
        )
    )

    run_text = (
        run_text[:data_argument_start]
        + patched_argument_block
        + run_text[data_argument_end:]
    )

RUN_FILE.write_text(
    run_text,
    encoding="utf-8",
)

# ============================================================
# 4. UTILISER TEST.CSV POUR LE VRAI TEST
# ============================================================

build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

old_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'val.csv')
"""

new_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'test.csv')
"""

if old_test_block in build_text:
    build_text = build_text.replace(
        old_test_block,
        new_test_block,
        1,
    )

# ============================================================
# 5. AJOUTER LE DATASET RHAD : 12 CLASSES
# ============================================================

rhad_branch = """
    elif args.data_set == 'RHAD':
        dataset = VideoClsDataset(
            anno_path=anno_path,
            data_root=args.data_root,
            mode=mode,
            clip_len=args.num_frames,
            frame_sample_rate=args.sampling_rate,
            num_segment=1,
            test_num_segment=args.test_num_segment,
            test_num_crop=args.test_num_crop,
            num_crop=1 if not test_mode else 3,
            keep_aspect_ratio=True,
            crop_size=args.input_size,
            short_side_size=args.short_side_size,
            new_height=256,
            new_width=320,
            sparse_sample=False,
            args=args,
        )
        nb_classes = 12
"""

unsupported_block = """    else:
        raise NotImplementedError('Unsupported Dataset')
"""

if "elif args.data_set == 'RHAD':" not in build_text:
    assert unsupported_block in build_text, (
        "Bloc Unsupported Dataset introuvable."
    )

    build_text = build_text.replace(
        unsupported_block,
        rhad_branch + "\n" + unsupported_block,
        1,
    )

BUILD_FILE.write_text(
    build_text,
    encoding="utf-8",
)

# ============================================================
# 6. VÉRIFIER LES PATCHS
# ============================================================

final_run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

final_build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DES MODIFICATIONS")
print("=" * 70)

checks = {
    "DeepSpeed optionnel":
        "except ImportError" in final_run_text,

    "RHAD autorisé dans les arguments":
        "'RHAD'" in final_run_text,

    "RHAD présent dans build.py":
        "elif args.data_set == 'RHAD':"
        in final_build_text,

    "RHAD possède 12 classes":
        "nb_classes = 12"
        in final_build_text,

    "Le test utilise test.csv":
        "anno_path = os.path.join("
        "args.data_path, 'test.csv')"
        in final_build_text,
}

for name, result in checks.items():
    print(f"{name:35s} :", result)
    assert result, name

# ============================================================
# 7. VÉRIFICATION DE LA SYNTAXE PYTHON
# ============================================================

subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(BUILD_FILE),
    ],
    check=True,
)

# Vérifier que les imports fonctionnent sans DeepSpeed
test_code = r"""
import run_class_finetuning
from dataset import build_dataset

print("Import run_class_finetuning : OK")
print("Import build_dataset : OK")
print("PATCH VIDEOMAE RHAD VALIDÉ")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print("\nSortie du test :")
print(result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test des fichiers modifiés a échoué."
    )

print("\nADAPTATION DU DÉPÔT TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 6
# Création du split identique à UniFormerV2 et VideoMamba
# + création d'alias vidéo uniques et sans espaces

from pathlib import Path
from collections import defaultdict, Counter
import csv
import json
import os
import random
import re
import shutil

# ============================================================
# 1. CONFIGURATION
# ============================================================

ORIGINAL_DATASET = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/"
    "training/Training"
)

# Dossier contenant des liens vers les vidéos originales.
# Les vidéos ne seront pas copiées.
VIDEOMAE_VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_videos"
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_annotations"
)

SEED = 42
CLIPS_PER_SOURCE = 5

VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mkv",
    ".mov",
    ".webm",
}

assert ORIGINAL_DATASET.exists(), (
    f"Dataset absent : {ORIGINAL_DATASET}"
)

# On recrée uniquement les dossiers de cette nouvelle expérience.
shutil.rmtree(
    VIDEOMAE_VIDEO_ROOT,
    ignore_errors=True,
)

shutil.rmtree(
    ANNOTATIONS_DIR,
    ignore_errors=True,
)

VIDEOMAE_VIDEO_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ANNOTATIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# 2. CLASSES ET LABELS
# ============================================================

classes = sorted(
    folder.name
    for folder in ORIGINAL_DATASET.iterdir()
    if folder.is_dir()
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

print("=" * 70)
print("CLASSES")
print("=" * 70)

print("Nombre de classes :", len(classes))
print("Mapping :", class_to_idx)

assert len(classes) == 12

expected_mapping = {
    "Jumping": 0,
    "child_running": 1,
    "climbing": 2,
    "collesion": 3,
    "electrical_appliances": 4,
    "falling": 5,
    "head_stuck": 6,
    "normal": 7,
    "objects_falling": 8,
    "pulling_curtains": 9,
    "scribbling": 10,
    "sharp_objects": 11,
}

assert class_to_idx == expected_mapping, (
    "Le mapping des classes diffère de celui utilisé "
    "par UniFormerV2 et VideoMamba."
)

# ============================================================
# 3. IDENTIFICATION DES GROUPES DE 5 CLIPS
# ============================================================

def extract_group_id(video_path: Path) -> str:
    """
    Exemple :
    1 (11).mp4 à 1 (15).mp4 appartiennent au même bloc.
    """

    stem = video_path.stem

    relative_parent = video_path.parent.relative_to(
        ORIGINAL_DATASET
    )

    match = re.match(
        r"^(.*?)\s*\((\d+)\)$",
        stem,
    )

    if match:
        prefix = match.group(1).strip()
        clip_number = int(match.group(2))

        block_number = (
            clip_number - 1
        ) // CLIPS_PER_SOURCE

        return (
            f"{relative_parent}/"
            f"{prefix}/block_{block_number:04d}"
        )

    # Un fichier ne suivant pas le format habituel devient
    # son propre groupe.
    return str(
        video_path.relative_to(ORIGINAL_DATASET)
    )

# ============================================================
# 4. REGROUPEMENT DES VIDÉOS PAR CLASSE
# ============================================================

groups_by_class = defaultdict(
    lambda: defaultdict(list)
)

all_original_videos = []

for class_name in classes:
    label = class_to_idx[class_name]

    class_directory = (
        ORIGINAL_DATASET / class_name
    )

    video_files = sorted(
        path
        for path in class_directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in VIDEO_EXTENSIONS
        )
    )

    for video_path in video_files:
        group_id = extract_group_id(video_path)

        groups_by_class[class_name][group_id].append(
            (video_path, label)
        )

        all_original_videos.append(video_path)

print("\nNombre total de vidéos :", len(all_original_videos))

assert len(all_original_videos) == 3248

# ============================================================
# 5. CRÉATION D'ALIAS UNIQUES, SANS ESPACES
# ============================================================

# Le tri rend la correspondance reproductible.
all_original_videos = sorted(
    all_original_videos,
    key=lambda path: str(
        path.relative_to(ORIGINAL_DATASET)
    ),
)

original_to_alias = {}

for global_index, original_path in enumerate(
    all_original_videos
):
    class_name = (
        original_path.relative_to(ORIGINAL_DATASET)
        .parts[0]
    )

    extension = original_path.suffix.lower()

    # Le nom est unique dans tout le dataset.
    # Il ne contient aucun espace.
    alias_name = (
        f"rhad_{global_index:05d}_"
        f"{class_name}{extension}"
    )

    alias_path = (
        VIDEOMAE_VIDEO_ROOT / alias_name
    )

    os.symlink(
        str(original_path),
        str(alias_path),
    )

    original_to_alias[
        str(original_path)
    ] = alias_name

print(
    "Nombre de liens symboliques créés :",
    len(original_to_alias),
)

assert len(original_to_alias) == 3248

# Vérifications importantes
alias_names = list(
    original_to_alias.values()
)

assert len(alias_names) == len(set(alias_names)), (
    "Certains alias ne sont pas uniques."
)

assert all(
    " " not in alias_name
    for alias_name in alias_names
), "Un alias contient encore un espace."

assert all(
    (VIDEOMAE_VIDEO_ROOT / alias_name).exists()
    for alias_name in alias_names
), "Au moins un lien symbolique est cassé."

# ============================================================
# 6. SPLIT PAR GROUPES : 80 / 10 / 10
# ============================================================

rng = random.Random(SEED)

train_samples = []
val_samples = []
test_samples = []

train_groups = set()
val_groups = set()
test_groups = set()

split_manifest = []

for class_name in classes:
    class_groups = list(
        groups_by_class[class_name].items()
    )

    # Même générateur et même ordre que pour les autres modèles.
    rng.shuffle(class_groups)

    number_of_groups = len(class_groups)

    if number_of_groups < 3:
        raise RuntimeError(
            f"{class_name} ne possède que "
            f"{number_of_groups} groupes."
        )

    number_of_test_groups = max(
        1,
        round(number_of_groups * 0.10),
    )

    number_of_val_groups = max(
        1,
        round(number_of_groups * 0.10),
    )

    while (
        number_of_test_groups
        + number_of_val_groups
        >= number_of_groups
    ):
        if number_of_test_groups > 1:
            number_of_test_groups -= 1
        elif number_of_val_groups > 1:
            number_of_val_groups -= 1
        else:
            break

    test_part = class_groups[
        :number_of_test_groups
    ]

    val_part = class_groups[
        number_of_test_groups:
        number_of_test_groups
        + number_of_val_groups
    ]

    train_part = class_groups[
        number_of_test_groups
        + number_of_val_groups:
    ]

    split_definitions = [
        (
            "train",
            train_part,
            train_samples,
            train_groups,
        ),
        (
            "val",
            val_part,
            val_samples,
            val_groups,
        ),
        (
            "test",
            test_part,
            test_samples,
            test_groups,
        ),
    ]

    for (
        split_name,
        split_groups,
        split_samples,
        split_group_set,
    ) in split_definitions:

        for group_id, group_samples in split_groups:
            split_group_set.add(group_id)

            split_samples.extend(
                group_samples
            )

            split_manifest.append({
                "split": split_name,
                "class_name": class_name,
                "label": class_to_idx[class_name],
                "group_id": group_id,
                "number_of_clips": len(group_samples),
            })

# Même mélange reproductible que précédemment.
rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

# ============================================================
# 7. VÉRIFICATION DES FUITES
# ============================================================

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

train_original_paths = {
    str(path)
    for path, _ in train_samples
}

val_original_paths = {
    str(path)
    for path, _ in val_samples
}

test_original_paths = {
    str(path)
    for path, _ in test_samples
}

assert train_original_paths.isdisjoint(
    val_original_paths
)

assert train_original_paths.isdisjoint(
    test_original_paths
)

assert val_original_paths.isdisjoint(
    test_original_paths
)

assert (
    len(train_samples)
    + len(val_samples)
    + len(test_samples)
    == 3248
)

# Les tailles doivent être strictement identiques
# à celles des deux autres modèles.
assert len(train_samples) == 2578
assert len(val_samples) == 335
assert len(test_samples) == 335

# ============================================================
# 8. ÉCRITURE DES ANNOTATIONS VIDEOMAE
# ============================================================

def write_videomae_annotations(
    destination: Path,
    samples,
):
    """
    Format officiel attendu :
    alias_video.mp4 label

    Un seul espace sépare le chemin du label.
    """

    with destination.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:

        for original_path, label in samples:
            alias_name = original_to_alias[
                str(original_path)
            ]

            file.write(
                f"{alias_name} {label}\n"
            )

write_videomae_annotations(
    ANNOTATIONS_DIR / "train.csv",
    train_samples,
)

write_videomae_annotations(
    ANNOTATIONS_DIR / "val.csv",
    val_samples,
)

write_videomae_annotations(
    ANNOTATIONS_DIR / "test.csv",
    test_samples,
)

# Mapping des labels
with (
    ANNOTATIONS_DIR / "label_mapping.csv"
).open(
    "w",
    encoding="utf-8",
    newline="",
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "label",
        "class_name",
    ])

    for class_name, label in class_to_idx.items():
        writer.writerow([
            label,
            class_name,
        ])

# Mapping alias -> chemin original
with (
    ANNOTATIONS_DIR / "path_mapping.csv"
).open(
    "w",
    encoding="utf-8",
    newline="",
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "alias",
        "original_path",
    ])

    for original_path, alias_name in sorted(
        original_to_alias.items(),
        key=lambda item: item[1],
    ):
        writer.writerow([
            alias_name,
            original_path,
        ])

# Manifeste des groupes
with (
    ANNOTATIONS_DIR / "split_groups.csv"
).open(
    "w",
    encoding="utf-8",
    newline="",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "split",
            "class_name",
            "label",
            "group_id",
            "number_of_clips",
        ],
    )

    writer.writeheader()
    writer.writerows(split_manifest)

# Configuration utile pour les prochaines cellules
configuration = {
    "dataset_root": str(VIDEOMAE_VIDEO_ROOT),
    "annotations_dir": str(ANNOTATIONS_DIR),
    "seed": SEED,
    "clips_per_source": CLIPS_PER_SOURCE,
    "train_size": len(train_samples),
    "val_size": len(val_samples),
    "test_size": len(test_samples),
    "number_of_classes": len(classes),
    "class_to_idx": class_to_idx,
}

with (
    ANNOTATIONS_DIR / "split_configuration.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        configuration,
        file,
        indent=2,
    )

# ============================================================
# 9. DISTRIBUTIONS PAR CLASSE
# ============================================================

def class_distribution(samples):
    counts = Counter(
        label
        for _, label in samples
    )

    return {
        classes[label]: counts[label]
        for label in range(len(classes))
    }

print("\n" + "=" * 70)
print("RÉSULTAT DU SPLIT VIDEOMAE")
print("=" * 70)

print("Train      :", len(train_samples))
print("Validation :", len(val_samples))
print("Test       :", len(test_samples))

print("\nGroupes train      :", len(train_groups))
print("Groupes validation :", len(val_groups))
print("Groupes test       :", len(test_groups))

print(
    "\nFuites train/val  :",
    len(train_groups & val_groups),
)

print(
    "Fuites train/test :",
    len(train_groups & test_groups),
)

print(
    "Fuites val/test   :",
    len(val_groups & test_groups),
)

for split_name, samples in [
    ("TRAIN", train_samples),
    ("VALIDATION", val_samples),
    ("TEST", test_samples),
]:
    print(f"\nRépartition {split_name} :")

    for class_name, count in class_distribution(
        samples
    ).items():
        print(
            f"{class_name:25s} : {count}"
        )

# ============================================================
# 10. VÉRIFICATION DU FORMAT DES CSV
# ============================================================

print("\n" + "=" * 70)
print("VÉRIFICATION DES ANNOTATIONS")
print("=" * 70)

for filename, expected_size in [
    ("train.csv", 2578),
    ("val.csv", 335),
    ("test.csv", 335),
]:
    path = ANNOTATIONS_DIR / filename

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    print(
        filename,
        ":",
        len(lines),
        "lignes",
    )

    assert len(lines) == expected_size

    # Chaque ligne doit contenir exactement :
    # nom_sans_espace + espace + label
    for line in lines:
        parts = line.split(" ")

        assert len(parts) == 2, (
            f"Ligne incorrecte : {line}"
        )

        alias_name, label_text = parts

        assert " " not in alias_name
        assert label_text.isdigit()
        assert (
            VIDEOMAE_VIDEO_ROOT / alias_name
        ).exists()

print("\nExemples train.csv :")

for line in (
    ANNOTATIONS_DIR / "train.csv"
).read_text(
    encoding="utf-8"
).splitlines()[:5]:
    print(line)

print("\nDossier vidéos alias :", VIDEOMAE_VIDEO_ROOT)
print("Dossier annotations  :", ANNOTATIONS_DIR)

print("\nSPLIT VIDEOMAE CRÉÉ ET VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 7A
# Télécharger le checkpoint préentraîné Kinetics-710

from pathlib import Path
import os
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

CHECKPOINT_DIR = Path(
    "/kaggle/working/videomae_pretrained"
)

EXPECTED_CHECKPOINT = (
    CHECKPOINT_DIR
    / "distill"
    / "vit_b_k710_dl_from_giant.pth"
)

assert Path(PY).exists(), (
    f"Environnement Python absent : {PY}"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

download_code = r"""
from pathlib import Path
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path(
    "/kaggle/working/videomae_pretrained"
)

checkpoint_path = Path(
    hf_hub_download(
        repo_id="OpenGVLab/VideoMAE2",
        filename="distill/vit_b_k710_dl_from_giant.pth",
        local_dir=str(CHECKPOINT_DIR),
    )
)

size_mib = checkpoint_path.stat().st_size / 1024**2

print("=" * 70)
print("CHECKPOINT PRÉENTRAÎNÉ TÉLÉCHARGÉ")
print("=" * 70)
print("Chemin :", checkpoint_path)
print("Taille :", f"{size_mib:.1f} MiB")

assert checkpoint_path.exists()
assert size_mib > 160, (
    "Le téléchargement semble incomplet."
)

print("TÉLÉCHARGEMENT TERMINÉ")
"""

environment = os.environ.copy()
environment["HF_HUB_DISABLE_TELEMETRY"] = "1"

result = subprocess.run(
    [
        PY,
        "-c",
        download_code,
    ],
    env=environment,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Le téléchargement du checkpoint "
        "Kinetics-710 a échoué."
    )

assert EXPECTED_CHECKPOINT.exists(), (
    f"Checkpoint introuvable : {EXPECTED_CHECKPOINT}"
)

print(
    "\nCheckpoint prêt pour la cellule 7B :",
    EXPECTED_CHECKPOINT,
)

In [ ]:
# VIDEOMAE — CELLULE 7B
# Vérification réelle du checkpoint déjà téléchargé

from pathlib import Path
from collections import Counter
import hashlib
import torch

CHECKPOINT_PATH = Path(
    "/kaggle/working/videomae_pretrained/"
    "distill/vit_b_k710_dl_from_giant.pth"
)

EXPECTED_SHA256 = (
    "8141a6955e0700d11bf15928fe6d61e5"
    "cfe482606fed8cfdddb1b922c0fd88ec"
)

# ============================================================
# 1. VÉRIFICATION DU FICHIER
# ============================================================

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint absent : {CHECKPOINT_PATH}"
)

size_bytes = CHECKPOINT_PATH.stat().st_size
size_decimal_mb = size_bytes / 1_000_000
size_mib = size_bytes / 1024**2

print("=" * 70)
print("FICHIER TÉLÉCHARGÉ")
print("=" * 70)

print("Chemin :", CHECKPOINT_PATH)
print("Taille décimale :", f"{size_decimal_mb:.1f} MB")
print("Taille binaire  :", f"{size_mib:.1f} MiB")

# Le fichier officiel fait environ 174 MB,
# donc une taille binaire autour de 165-166 MiB est normale.
assert size_mib > 160, (
    "Le fichier semble réellement incomplet."
)

# ============================================================
# 2. VÉRIFICATION SHA-256
# ============================================================

print("\nCalcul du SHA-256...")

sha256 = hashlib.sha256()

with CHECKPOINT_PATH.open("rb") as file:
    while True:
        chunk = file.read(8 * 1024 * 1024)

        if not chunk:
            break

        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

print("SHA-256 obtenu  :", actual_sha256)
print("SHA-256 attendu :", EXPECTED_SHA256)

assert actual_sha256 == EXPECTED_SHA256, (
    "Le SHA-256 ne correspond pas au fichier officiel."
)

print("Intégrité du téléchargement : VALIDÉE")

# ============================================================
# 3. CHARGEMENT DU CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("\n" + "=" * 70)
print("CONTENU DU CHECKPOINT")
print("=" * 70)

print("Type :", type(checkpoint))

if not isinstance(checkpoint, dict):
    raise TypeError(
        "Le checkpoint n'est pas un dictionnaire."
    )

print("Clés principales :", list(checkpoint.keys()))

if "model" in checkpoint:
    state_dict = checkpoint["model"]

elif "module" in checkpoint:
    state_dict = checkpoint["module"]

elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]

else:
    state_dict = checkpoint

print("Nombre de tenseurs :", len(state_dict))

assert len(state_dict) > 100

# ============================================================
# 4. VÉRIFICATION DE L'ARCHITECTURE
# ============================================================

head_keys = [
    key
    for key in state_dict
    if key.endswith("head.weight")
]

print("Clés de tête trouvées :", head_keys)

assert head_keys, (
    "Aucune tête de classification trouvée."
)

head_key = head_keys[0]
head_shape = tuple(state_dict[head_key].shape)

print("Forme de la tête :", head_shape)

assert head_shape == (710, 768), (
    "Le checkpoint ne correspond pas à "
    "ViT-Base Kinetics-710."
)

patch_keys = [
    key
    for key in state_dict
    if key.endswith("patch_embed.proj.weight")
]

if patch_keys:
    patch_key = patch_keys[0]

    print(
        "Forme du patch embedding :",
        tuple(state_dict[patch_key].shape),
    )

dtype_counts = Counter(
    str(tensor.dtype)
    for tensor in state_dict.values()
    if torch.is_tensor(tensor)
)

print("Types des poids :", dict(dtype_counts))

# ============================================================
# 5. SAUVEGARDER LE CHEMIN
# ============================================================

PATH_FILE = Path(
    "/kaggle/working/"
    "videomae_pretrained_checkpoint_path.txt"
)

PATH_FILE.write_text(
    str(CHECKPOINT_PATH),
    encoding="utf-8",
)

print("\nChemin enregistré dans :", PATH_FILE)

print("\nCHECKPOINT KINETICS-710 VALIDÉ")
print("ANCIEN CHECKPOINT-39 NON UTILISÉ")

In [ ]:
# VIDEOMAE — CELLULE 8
# Smoke test du dataset, du checkpoint et du modèle sur GPU

from pathlib import Path
import os
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

REPO_DIR = Path(
    "/kaggle/working/VideoMAEv2"
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_videos"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/videomae_pretrained/"
    "distill/vit_b_k710_dl_from_giant.pth"
)

assert Path(PY).exists(), PY
assert REPO_DIR.exists(), REPO_DIR
assert ANNOTATIONS_DIR.exists(), ANNOTATIONS_DIR
assert VIDEO_ROOT.exists(), VIDEO_ROOT
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

test_code = r"""
from pathlib import Path
from types import SimpleNamespace
import gc

import torch
import timm

# Enregistre les architectures VideoMAE dans timm
import models

from dataset import build_dataset

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_videos"
)

CHECKPOINT_PATH = Path(
    "/kaggle/working/videomae_pretrained/"
    "distill/vit_b_k710_dl_from_giant.pth"
)

# ============================================================
# 1. PARAMÈTRES DU DATASET
# ============================================================

args = SimpleNamespace(
    data_path=str(ANNOTATIONS_DIR),
    data_root=str(VIDEO_ROOT),
    data_set="RHAD",
    nb_classes=12,

    num_frames=16,
    sampling_rate=4,
    sparse_sample=False,

    input_size=224,
    short_side_size=224,

    # Évaluation finale :
    # 2 segments temporels × 3 crops spatiaux
    test_num_segment=2,
    test_num_crop=3,

    # Augmentations utilisées par le train loader
    num_sample=1,
    aa="rand-m7-n4-mstd0.5-inc1",
    train_interpolation="bicubic",

    reprob=0.25,
    remode="pixel",
    recount=1,
)

print("=" * 70)
print("1. CRÉATION DES DATASETS")
print("=" * 70)

train_dataset, train_classes = build_dataset(
    is_train=True,
    test_mode=False,
    args=args,
)

val_dataset, val_classes = build_dataset(
    is_train=False,
    test_mode=False,
    args=args,
)

test_dataset, test_classes = build_dataset(
    is_train=False,
    test_mode=True,
    args=args,
)

print("Train      :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test views :", len(test_dataset))

print(
    "Test attendu :",
    "335 vidéos × 2 segments × 3 crops =",
    335 * 2 * 3,
)

assert train_classes == 12
assert val_classes == 12
assert test_classes == 12

assert len(train_dataset) == 2578
assert len(val_dataset) == 335
assert len(test_dataset) == 2010

# ============================================================
# 2. LECTURE DE VRAIES VIDÉOS
# ============================================================

print("\n" + "=" * 70)
print("2. LECTURE DES ÉCHANTILLONS")
print("=" * 70)

train_item = train_dataset[0]
val_item = val_dataset[0]
test_item = test_dataset[0]

train_video = train_item[0]
train_label = int(train_item[1])

val_video = val_item[0]
val_label = int(val_item[1])
val_name = val_item[2]

test_video = test_item[0]
test_label = int(test_item[1])
test_name = test_item[2]
test_segment = int(test_item[3])
test_crop = int(test_item[4])

print(
    "Train :",
    tuple(train_video.shape),
    "label =",
    train_label,
)

print(
    "Val   :",
    tuple(val_video.shape),
    "label =",
    val_label,
    "nom =",
    val_name,
)

print(
    "Test  :",
    tuple(test_video.shape),
    "label =",
    test_label,
    "nom =",
    test_name,
    "segment =",
    test_segment,
    "crop =",
    test_crop,
)

expected_shape = (
    3,
    16,
    224,
    224,
)

assert tuple(train_video.shape) == expected_shape
assert tuple(val_video.shape) == expected_shape
assert tuple(test_video.shape) == expected_shape

assert train_video.dtype == torch.float32
assert val_video.dtype == torch.float32
assert test_video.dtype == torch.float32

assert torch.isfinite(train_video).all()
assert torch.isfinite(val_video).all()
assert torch.isfinite(test_video).all()

assert 0 <= train_label < 12
assert 0 <= val_label < 12
assert 0 <= test_label < 12

print("LECTURE ET TRANSFORMATIONS VIDÉO VALIDÉES")

# ============================================================
# 3. CRÉATION DU MODÈLE RHAD
# ============================================================

print("\n" + "=" * 70)
print("3. CRÉATION DU MODÈLE")
print("=" * 70)

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_rate=0.0,
    drop_path_rate=0.1,
    attn_drop_rate=0.0,
    head_drop_rate=0.0,
    use_mean_pooling=True,
    init_scale=0.001,
)

print(
    "Paramètres du modèle :",
    f"{sum(p.numel() for p in model.parameters()):,}",
)

print(
    "Tête RHAD :",
    tuple(model.head.weight.shape),
)

assert tuple(
    model.head.weight.shape
) == (12, 768)

# ============================================================
# 4. CHARGEMENT DU CHECKPOINT KINETICS-710
# ============================================================

print("\n" + "=" * 70)
print("4. CHARGEMENT DES POIDS KINETICS-710")
print("=" * 70)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

state_dict = checkpoint["module"]

print(
    "Tête originale Kinetics :",
    tuple(state_dict["head.weight"].shape),
)

# La tête Kinetics possède 710 sorties.
# Elle doit être retirée pour conserver la nouvelle tête RHAD.
state_dict = dict(state_dict)

state_dict.pop(
    "head.weight",
    None,
)

state_dict.pop(
    "head.bias",
    None,
)

load_result = model.load_state_dict(
    state_dict,
    strict=False,
)

missing_keys = list(
    load_result.missing_keys
)

unexpected_keys = list(
    load_result.unexpected_keys
)

print("Clés manquantes :", missing_keys)
print("Clés inattendues :", unexpected_keys)

allowed_missing = {
    "head.weight",
    "head.bias",
}

invalid_missing = (
    set(missing_keys)
    - allowed_missing
)

assert not invalid_missing, (
    "Clés importantes manquantes : "
    f"{sorted(invalid_missing)}"
)

assert not unexpected_keys, (
    "Clés inattendues : "
    f"{unexpected_keys}"
)

print(
    "Backbone Kinetics chargé ; "
    "nouvelle tête RHAD conservée."
)

# ============================================================
# 5. VRAI PASSAGE SUR LE GPU
# ============================================================

print("\n" + "=" * 70)
print("5. FORWARD PASS SUR GPU")
print("=" * 70)

assert torch.cuda.is_available()

device = torch.device("cuda:0")

model = model.to(device)
model.eval()

# Deux vidéos de validation pour tester un vrai batch.
second_val_video = val_dataset[1][0]

batch = torch.stack([
    val_video,
    second_val_video,
]).to(
    device,
    non_blocking=True,
)

print("Batch envoyé au GPU :", tuple(batch.shape))

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        logits = model(batch)

print("Sortie du modèle :", tuple(logits.shape))
print("Type de sortie :", logits.dtype)

assert tuple(logits.shape) == (2, 12)
assert torch.isfinite(logits).all()

probabilities = torch.softmax(
    logits.float(),
    dim=1,
)

predictions = probabilities.argmax(
    dim=1,
)

confidences = probabilities.max(
    dim=1,
).values

print(
    "Prédictions initiales :",
    predictions.cpu().tolist(),
)

print(
    "Confiances initiales :",
    [
        round(value, 4)
        for value in confidences.cpu().tolist()
    ],
)

print(
    "Mémoire GPU maximale :",
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} Go",
)

del model
del batch
del logits
del checkpoint

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("SMOKE TEST VIDEOMAE ENTIÈREMENT VALIDÉ")
print("=" * 70)
print("Le dataset, les poids et le GPU sont prêts.")
print("Aucun entraînement n'a encore été lancé.")
"""

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    env=environment,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("\nMESSAGES SYSTÈME :")
    print(result.stderr)

print("\nCode de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le smoke test VideoMAE a échoué."
    )

In [ ]:
# VIDEOMAE — CELLULE 9
# Conserver uniquement checkpoint-latest et checkpoint-best

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

UTILS_FILE = Path(
    "/kaggle/working/VideoMAEv2/utils.py"
)

assert Path(PY).exists(), PY
assert UTILS_FILE.exists(), UTILS_FILE

# ============================================================
# 1. SAUVEGARDE DU FICHIER ORIGINAL
# ============================================================

BACKUP_FILE = UTILS_FILE.with_suffix(
    ".py.original"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        UTILS_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. MODIFICATION DU NOM DES CHECKPOINTS
# ============================================================

text = UTILS_FILE.read_text(
    encoding="utf-8"
)

old_code = """    if loss_scaler is not None:
        checkpoint_paths = [output_dir / ('checkpoint-%s.pth' % epoch_name)]
"""

new_code = """    if loss_scaler is not None:
        # Le meilleur modèle reste dans checkpoint-best.pth.
        # Tous les checkpoints périodiques écrasent checkpoint-latest.pth
        # afin de ne pas créer des dizaines de fichiers volumineux.
        if epoch_name == "best":
            checkpoint_filename = "checkpoint-best.pth"
        else:
            checkpoint_filename = "checkpoint-latest.pth"

        checkpoint_paths = [
            output_dir / checkpoint_filename
        ]
"""

if old_code in text:
    text = text.replace(
        old_code,
        new_code,
        1,
    )
    UTILS_FILE.write_text(
        text,
        encoding="utf-8",
    )
    print("Patch appliqué.")
elif new_code in text:
    print("Patch déjà appliqué.")
else:
    raise RuntimeError(
        "Le bloc de sauvegarde attendu n'a pas été trouvé."
    )

# ============================================================
# 3. VÉRIFICATIONS
# ============================================================

final_text = UTILS_FILE.read_text(
    encoding="utf-8"
)

checks = {
    "checkpoint-latest présent":
        'checkpoint_filename = "checkpoint-latest.pth"'
        in final_text,

    "checkpoint-best présent":
        'checkpoint_filename = "checkpoint-best.pth"'
        in final_text,

    "ancien fichier numérique supprimé":
        "checkpoint_paths = [output_dir / "
        "('checkpoint-%s.pth' % epoch_name)]"
        not in final_text,
}

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

for name, result in checks.items():
    print(f"{name:40s} :", result)
    assert result, name

# Vérification de la syntaxe
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(UTILS_FILE),
    ],
    check=True,
)

# Vérification par import
test_code = r"""
import utils

print("Import utils : OK")
print("GESTION DES CHECKPOINTS VALIDÉE")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd="/kaggle/working/VideoMAEv2",
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test du patch checkpoints a échoué."
    )

print("\nPRÉPARATION DES CHECKPOINTS TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 10A
# Corriger sampler_test

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"
RUN_FILE = Path(
    "/kaggle/working/VideoMAEv2/run_class_finetuning.py"
)

assert RUN_FILE.exists()
assert Path(PY).exists()

backup = RUN_FILE.with_name(
    "run_class_finetuning.py.before_sampler_patch"
)

if not backup.exists():
    shutil.copy2(RUN_FILE, backup)

text = RUN_FILE.read_text(encoding="utf-8")

old_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
"""

new_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
        sampler_test = torch.utils.data.DistributedSampler(
            dataset_test,
            num_replicas=num_tasks,
            rank=global_rank,
            shuffle=False,
        )
"""

start = text.index("    if args.dist_eval:")
end = text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
section = text[start:end]
else_section = section.split("    else:", 1)[-1]

if "sampler_test =" not in else_section:
    assert old_block in section
    section = section.replace(old_block, new_block, 1)
    text = text[:start] + section + text[end:]
    RUN_FILE.write_text(text, encoding="utf-8")
    print("Patch sampler_test appliqué.")
else:
    print("Patch sampler_test déjà présent.")

final_text = RUN_FILE.read_text(encoding="utf-8")
start = final_text.index("    if args.dist_eval:")
end = final_text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
final_section = final_text[start:end]

assert "sampler_test =" in final_section.split("    else:", 1)[-1]

subprocess.run(
    [PY, "-m", "py_compile", str(RUN_FILE)],
    check=True,
)

print("SAMPLER_TEST VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 10B
# Remplacer le bfloat16 incompatible avec les T4 par du float16

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

ENGINE_FILE = Path(
    "/kaggle/working/VideoMAEv2/"
    "engine_for_finetuning.py"
)

assert Path(PY).exists(), PY
assert ENGINE_FILE.exists(), ENGINE_FILE

# ============================================================
# 1. SAUVEGARDE
# ============================================================

BACKUP_FILE = ENGINE_FILE.with_name(
    "engine_for_finetuning.py.before_fp16_patch"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        ENGINE_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. INSPECTION AVANT MODIFICATION
# ============================================================

text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_before = text.count(
    "torch.bfloat16"
)

print(
    "Occurrences torch.bfloat16 avant patch :",
    bf16_count_before,
)

if bf16_count_before == 0:
    print(
        "Aucune occurrence BF16 : "
        "le patch semble déjà appliqué."
    )
else:
    # Le fichier concerne uniquement le fine-tuning.
    # Toutes ses opérations AMP doivent utiliser FP16 sur T4.
    text = text.replace(
        "torch.bfloat16",
        "torch.float16",
    )

    ENGINE_FILE.write_text(
        text,
        encoding="utf-8",
    )

    print(
        "Toutes les occurrences BF16 ont été "
        "remplacées par FP16."
    )

# ============================================================
# 3. VÉRIFICATION
# ============================================================

final_text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_after = final_text.count(
    "torch.bfloat16"
)

fp16_count = final_text.count(
    "torch.float16"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

print(
    "Occurrences torch.bfloat16 après patch :",
    bf16_count_after,
)

print(
    "Occurrences torch.float16 :",
    fp16_count,
)

assert bf16_count_after == 0, (
    "Il reste encore une opération en bfloat16."
)

assert fp16_count >= 1, (
    "Aucune opération float16 n'a été trouvée."
)

# Vérifier la syntaxe Python
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(ENGINE_FILE),
    ],
    check=True,
)

# ============================================================
# 4. TEST AMP RÉEL SUR LE T4
# ============================================================

test_code = r"""
import torch

assert torch.cuda.is_available()

x = torch.randn(
    256,
    256,
    device="cuda",
)

with torch.cuda.amp.autocast(
    dtype=torch.float16
):
    y = x @ x.T

print("GPU :", torch.cuda.get_device_name(0))
print("Type de sortie :", y.dtype)
print("Forme :", tuple(y.shape))
print("Valeurs finies :", torch.isfinite(y).all().item())

assert y.dtype == torch.float16
assert torch.isfinite(y).all()

print("AMP FLOAT16 VALIDÉ SUR TESLA T4")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test FP16 sur le GPU a échoué."
    )

print("\nPATCH FP16 VIDEOMAE VALIDÉ")

In [ ]:
# ============================================================
# VIDEOMAE — CELLULE 11A
# Restaurer précisément les checkpoints sauvegardés sur Kaggle
# ============================================================

from pathlib import Path
import gc
import shutil
import subprocess
import torch

PY = "/kaggle/working/vmae_env/bin/python"

REPO_DIR = Path(
    "/kaggle/working/VideoMAEv2"
)

RUN_FILE = (
    REPO_DIR / "run_class_finetuning.py"
)

ENGINE_FILE = (
    REPO_DIR / "engine_for_finetuning.py"
)

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

LATEST_DESTINATION = (
    OUTPUT_DIR / "checkpoint-latest.pth"
)

BEST_DESTINATION = (
    OUTPUT_DIR / "checkpoint-best.pth"
)

OLD_BEST_BACKUP = (
    OUTPUT_DIR
    / "checkpoint-best-before-resume.pth"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# 1. SÉLECTIONNER DIRECTEMENT LES DEUX BONS CHECKPOINTS
# ============================================================

LATEST_SOURCE = Path(
    "/kaggle/input/datasets/"
    "nisrineamalkhnifass/"
    "videomae-resume-epoch31/"
    "checkpoint-latest.pth"
)

BEST_SOURCE = Path(
    "/kaggle/input/datasets/"
    "nisrineamalkhnifass/"
    "videomae-best-and-logs-epoch31/"
    "checkpoint-best.pth"
)

print("=" * 70)
print("CHECKPOINTS VIDEOMAE SÉLECTIONNÉS")
print("=" * 70)

print("\nCheckpoint latest :")
print(LATEST_SOURCE)

print("\nCheckpoint best :")
print(BEST_SOURCE)

assert LATEST_SOURCE.exists(), (
    f"Checkpoint latest introuvable : {LATEST_SOURCE}"
)

assert BEST_SOURCE.exists(), (
    f"Checkpoint best introuvable : {BEST_SOURCE}"
)

print(
    "\nTaille latest :",
    f"{LATEST_SOURCE.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "Taille best   :",
    f"{BEST_SOURCE.stat().st_size / 1024**2:.2f} MiB",
)

# ============================================================
# 2. COPIER LES CHECKPOINTS DANS LE DOSSIER DE SORTIE
# ============================================================

shutil.copy2(
    LATEST_SOURCE,
    LATEST_DESTINATION,
)

shutil.copy2(
    BEST_SOURCE,
    BEST_DESTINATION,
)

# Copie protégée de l'ancien meilleur modèle.
shutil.copy2(
    BEST_SOURCE,
    OLD_BEST_BACKUP,
)

print("\n" + "=" * 70)
print("CHECKPOINTS COPIÉS")
print("=" * 70)

print("Latest :", LATEST_DESTINATION)
print("Best   :", BEST_DESTINATION)
print("Ancien best protégé :", OLD_BEST_BACKUP)

# ============================================================
# 3. VÉRIFIER CHECKPOINT-LATEST
# ============================================================

latest_checkpoint = torch.load(
    LATEST_DESTINATION,
    map_location="cpu",
    weights_only=False,
)

latest_epoch = latest_checkpoint.get(
    "epoch"
)

assert isinstance(latest_epoch, int), (
    "L'epoch du checkpoint latest est invalide."
)

# Nous avons précisément sauvegardé l'epoch 31.
assert latest_epoch == 31, (
    f"Le checkpoint devrait contenir l'epoch 31, "
    f"mais il contient l'epoch {latest_epoch}."
)

next_epoch = latest_epoch + 1

assert next_epoch == 32
assert next_epoch < 40

latest_model = latest_checkpoint.get(
    "model",
    {},
)

assert "head.weight" in latest_model, (
    "La couche head.weight est absente."
)

assert tuple(
    latest_model["head.weight"].shape
) == (12, 768), (
    "La tête du modèle ne correspond pas aux 12 classes RHAD."
)

assert "optimizer" in latest_checkpoint, (
    "L'état de l'optimiseur est absent."
)

assert (
    "scaler" in latest_checkpoint
    or "loss_scaler" in latest_checkpoint
), (
    "L'état du scaler FP16 est absent."
)

print("\n" + "=" * 70)
print("CHECKPOINT LATEST VALIDÉ")
print("=" * 70)

print("Dernière epoch terminée :", latest_epoch)
print("Reprise prévue à l'epoch :", next_epoch)

print(
    "Tête RHAD :",
    tuple(latest_model["head.weight"].shape),
)

print("Optimiseur présent : True")
print("Scaler FP16 présent : True")

del latest_model
del latest_checkpoint
gc.collect()

# ============================================================
# 4. VÉRIFIER CHECKPOINT-BEST
# ============================================================

best_checkpoint = torch.load(
    BEST_DESTINATION,
    map_location="cpu",
    weights_only=False,
)

best_epoch = best_checkpoint.get(
    "epoch"
)

best_model = best_checkpoint.get(
    "model",
    {},
)

assert "head.weight" in best_model, (
    "La couche head.weight est absente du checkpoint best."
)

assert tuple(
    best_model["head.weight"].shape
) == (12, 768)

print("\n" + "=" * 70)
print("ANCIEN CHECKPOINT BEST PROTÉGÉ")
print("=" * 70)

print("Epoch du meilleur checkpoint :", best_epoch)
print("Checkpoint best actif :", BEST_DESTINATION)
print("Copie protégée        :", OLD_BEST_BACKUP)

del best_model
del best_checkpoint
gc.collect()

# ============================================================
# 5. VÉRIFICATIONS TECHNIQUES DU NOTEBOOK
# ============================================================

for path in [
    Path(PY),
    RUN_FILE,
    ENGINE_FILE,
    LATEST_DESTINATION,
    BEST_DESTINATION,
    OLD_BEST_BACKUP,
]:
    assert path.exists(), (
        f"Élément absent : {path}"
    )

run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

engine_text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

assert run_text.count(
    "sampler_test ="
) >= 2, (
    "La correction sampler_test manque."
)

assert "torch.bfloat16" not in engine_text, (
    "La correction FP16 manque."
)

assert "torch.float16" in engine_text

subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(ENGINE_FILE),
    ],
    check=True,
)

print("\n" + "=" * 70)
print("REPRISE VIDEOMAE PRÉPARÉE ET VALIDÉE")
print("=" * 70)

print("Dernière epoch enregistrée :", latest_epoch)
print("Prochaine epoch            :", next_epoch)
print("Dernière epoch prévue      : 39")

print(
    "\nTU PEUX MAINTENANT LANCER LA CELLULE 11B"
)

In [ ]:
# VIDEOMAE — CELLULE 11B
# Reprendre l'entraînement depuis checkpoint-latest.pth

from pathlib import Path
import datetime
import os
import subprocess
import torch

PY = "/kaggle/working/vmae_env/bin/python"
REPO_DIR = Path("/kaggle/working/VideoMAEv2")

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_videos"
)

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

LATEST = OUTPUT_DIR / "checkpoint-latest.pth"

for path in [
    Path(PY),
    REPO_DIR,
    ANNOTATIONS_DIR / "train.csv",
    ANNOTATIONS_DIR / "val.csv",
    ANNOTATIONS_DIR / "test.csv",
    VIDEO_ROOT,
    LATEST,
]:
    assert path.exists(), path

# Vérifier qu'un autre entraînement ne tourne pas déjà.
process_check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps aux | grep run_class_finetuning.py "
        "| grep -v grep || true",
    ],
    text=True,
    capture_output=True,
)

if process_check.stdout.strip():
    print(process_check.stdout)
    raise RuntimeError(
        "Un entraînement VideoMAE est déjà actif."
    )

checkpoint = torch.load(
    LATEST,
    map_location="cpu",
    weights_only=False,
)

saved_epoch = checkpoint.get("epoch")
assert isinstance(saved_epoch, int)
next_epoch = saved_epoch + 1
del checkpoint

assert next_epoch < 40

print("=" * 70)
print("REPRISE VIDEOMAE")
print("=" * 70)
print("Dernière epoch terminée :", saved_epoch)
print("Reprise à l'epoch       :", next_epoch)
print("Dernière epoch prévue   : 39")
print("Checkpoint utilisé      :", LATEST)

command = [
    PY,
    "-m",
    "torch.distributed.run",
    "--nproc_per_node=2",
    "--master_port=29522",
    "run_class_finetuning.py",

    "--model", "vit_base_patch16_224",
    "--nb_classes", "12",
    "--use_mean_pooling",
    "--init_scale", "0.001",

    "--data_set", "RHAD",
    "--data_path", str(ANNOTATIONS_DIR),
    "--data_root", str(VIDEO_ROOT),

    "--input_size", "224",
    "--short_side_size", "224",
    "--num_frames", "16",
    "--num_segments", "1",
    "--tubelet_size", "2",
    "--sampling_rate", "4",

    "--epochs", "40",
    "--start_epoch", str(next_epoch),

    "--batch_size", "2",
    "--update_freq", "1",
    "--num_workers", "2",
    "--seed", "42",

    "--opt", "adamw",
    "--lr", "1e-3",
    "--min_lr", "1e-6",
    "--weight_decay", "0.05",
    "--layer_decay", "0.75",
    "--warmup_epochs", "5",
    "--clip_grad", "5.0",
    "--opt_betas", "0.9", "0.999",
    "--drop_path", "0.1",

    "--num_sample", "1",
    "--smoothing", "0.1",
    "--mixup", "0.8",
    "--cutmix", "1.0",
    "--mixup_prob", "1.0",
    "--mixup_switch_prob", "0.5",
    "--color_jitter", "0.4",
    "--aa", "rand-m7-n4-mstd0.5-inc1",
    "--reprob", "0.25",
    "--remode", "pixel",
    "--recount", "1",

    "--test_num_segment", "2",
    "--test_num_crop", "3",

    "--output_dir", str(OUTPUT_DIR),
    "--log_dir", str(OUTPUT_DIR),
    "--save_ckpt",
    "--save_ckpt_freq", "1",

    "--resume", str(LATEST),
    "--no_auto_resume",
]

assert "--finetune" not in command
assert "--resume" in command

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["OMP_NUM_THREADS"] = "1"
environment["PYTHONUNBUFFERED"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"

timestamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

resume_log = OUTPUT_DIR / (
    f"training_console_resume_epoch_"
    f"{next_epoch}_{timestamp}.log"
)

print("Log de reprise :", resume_log)
print("=" * 70 + "\n")

with resume_log.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=str(REPO_DIR),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"La reprise s'est arrêtée avec le code {return_code}."
    )

print("LES 40 EPOCHS SONT TERMINÉES")


In [ ]:
# ============================================================
# VIDEOMAE — VÉRIFIER LES FICHIERS APRÈS L'ENTRAÎNEMENT
# ============================================================

from pathlib import Path
import torch
import gc

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

print("=" * 80)
print("FICHIERS VIDEOMAE DISPONIBLES")
print("=" * 80)

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(
            path.relative_to(OUTPUT_DIR),
            f"— {path.stat().st_size / 1024**2:.2f} MiB"
        )

for checkpoint_name in [
    "checkpoint-best.pth",
    "checkpoint-latest.pth",
]:
    checkpoint_path = OUTPUT_DIR / checkpoint_name

    print("\n" + "=" * 80)
    print(checkpoint_name)
    print("=" * 80)

    assert checkpoint_path.exists(), checkpoint_path

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    print("Epoch enregistrée :", checkpoint.get("epoch"))
    print("Clés :", list(checkpoint.keys()))

    model_weights = checkpoint.get("model", {})
    print(
        "Tête finale :",
        tuple(model_weights["head.weight"].shape)
    )

    del checkpoint
    gc.collect()

print("\nVÉRIFICATION TERMINÉE")

In [ ]:
# ============================================================
# VIDEOMAE — IDENTIFIER LE MEILLEUR CHECKPOINT FINAL
# ============================================================

from pathlib import Path
import hashlib
import json
import torch
import gc

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

CURRENT_BEST = OUTPUT_DIR / "checkpoint-best.pth"
OLD_BEST = OUTPUT_DIR / "checkpoint-best-before-resume.pth"
LATEST = OUTPUT_DIR / "checkpoint-latest.pth"
LOG_FILE = OUTPUT_DIR / "log.txt"


def hash_model_weights(checkpoint_path: Path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    state_dict = checkpoint["model"]
    hasher = hashlib.sha256()

    for key in sorted(state_dict.keys()):
        tensor = (
            state_dict[key]
            .detach()
            .cpu()
            .contiguous()
        )

        hasher.update(key.encode("utf-8"))
        hasher.update(tensor.numpy().tobytes())

    epoch = checkpoint.get("epoch")

    del state_dict
    del checkpoint
    gc.collect()

    return hasher.hexdigest(), epoch


print("=" * 80)
print("COMPARAISON DES CHECKPOINTS BEST")
print("=" * 80)

current_hash, current_epoch = hash_model_weights(
    CURRENT_BEST
)

old_hash, old_epoch = hash_model_weights(
    OLD_BEST
)

print("\nCheckpoint-best actuel")
print("Epoch :", current_epoch)
print("Hash  :", current_hash)

print("\nCheckpoint-best avant reprise")
print("Epoch :", old_epoch)
print("Hash  :", old_hash)

if current_hash == old_hash:
    print(
        "\nLes deux checkpoints BEST "
        "contiennent exactement les mêmes poids."
    )
else:
    print(
        "\nATTENTION : Le checkpoint BEST a changé "
        "pendant les epochs 32 à 39."
    )


# ============================================================
# Lire les résultats du fichier log.txt
# ============================================================

print("\n" + "=" * 80)
print("HISTORIQUE DES VALIDATIONS")
print("=" * 80)

records = []

if LOG_FILE.exists():
    for line in LOG_FILE.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
            records.append(record)
        except json.JSONDecodeError:
            continue

print("Nombre de lignes JSON :", len(records))

validation_rows = []

for record in records:
    epoch = record.get("epoch")

    accuracy = None

    for key in [
        "test_acc1",
        "val_acc1",
        "validation_acc1",
        "test_acc",
        "val_acc",
    ]:
        if key in record:
            accuracy = record[key]
            break

    if accuracy is not None:
        validation_rows.append(
            {
                "epoch": epoch,
                "accuracy": float(accuracy),
                "loss": record.get(
                    "test_loss",
                    record.get("val_loss"),
                ),
            }
        )

for row in validation_rows:
    print(
        f"Epoch {str(row['epoch']):>2} "
        f"→ Acc1 = {row['accuracy']:.4f} "
        f"| loss = {row['loss']}"
    )

if validation_rows:
    best_row = max(
        validation_rows,
        key=lambda row: row["accuracy"],
    )

    print("\nMeilleur résultat trouvé dans log.txt :")
    print("Epoch    :", best_row["epoch"])
    print("Accuracy :", best_row["accuracy"])
    print("Loss     :", best_row["loss"])
else:
    print(
        "\nAucun résultat de validation JSON "
        "trouvé dans log.txt."
    )


# ============================================================
# Examiner les fichiers bruts 0.txt et 1.txt
# ============================================================

print("\n" + "=" * 80)
print("FICHIERS BRUTS DU TEST")
print("=" * 80)

for filename in ["0.txt", "1.txt"]:
    path = OUTPUT_DIR / filename

    if not path.exists():
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    print(f"\n{filename}")
    print("Nombre de lignes :", len(lines))
    print("Premières lignes :")

    for line in lines[:3]:
        print(line[:300])

print("\nINSPECTION TERMINÉE")

In [ ]:
# ============================================================
# VIDEOMAE — CELLULE 12A
# Préparer les évaluations validation et test
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

PY = "/kaggle/working/vmae_env/bin/python"

REPO_DIR = Path(
    "/kaggle/working/VideoMAEv2"
)

RUN_FILE = (
    REPO_DIR / "run_class_finetuning.py"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_videos"
)

SOURCE_ANNOTATIONS = Path(
    "/kaggle/working/rhad_videomae_annotations"
)

TRAIN_OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

BEST_CHECKPOINT = (
    TRAIN_OUTPUT_DIR / "checkpoint-best.pth"
)

FINAL_EVAL_DIR = Path(
    "/kaggle/working/videomae_final_evaluation"
)

VAL_ANNOTATIONS_DIR = (
    FINAL_EVAL_DIR / "annotations_validation"
)

TEST_ANNOTATIONS_DIR = (
    FINAL_EVAL_DIR / "annotations_test"
)

VAL_OUTPUT_DIR = (
    FINAL_EVAL_DIR / "validation_raw"
)

TEST_OUTPUT_DIR = (
    FINAL_EVAL_DIR / "test_raw"
)

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

for path in [
    Path(PY),
    RUN_FILE,
    VIDEO_ROOT,
    SOURCE_ANNOTATIONS / "train.csv",
    SOURCE_ANNOTATIONS / "val.csv",
    SOURCE_ANNOTATIONS / "test.csv",
    BEST_CHECKPOINT,
]:
    assert path.exists(), path

# ------------------------------------------------------------
# Nettoyer uniquement les anciens dossiers d'évaluation
# ------------------------------------------------------------

for directory in [
    VAL_ANNOTATIONS_DIR,
    TEST_ANNOTATIONS_DIR,
    VAL_OUTPUT_DIR,
    TEST_OUTPUT_DIR,
]:
    if directory.exists():
        shutil.rmtree(directory)

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# Validation : utiliser val.csv comme jeu évalué
# ------------------------------------------------------------

shutil.copy2(
    SOURCE_ANNOTATIONS / "train.csv",
    VAL_ANNOTATIONS_DIR / "train.csv",
)

shutil.copy2(
    SOURCE_ANNOTATIONS / "val.csv",
    VAL_ANNOTATIONS_DIR / "val.csv",
)

shutil.copy2(
    SOURCE_ANNOTATIONS / "val.csv",
    VAL_ANNOTATIONS_DIR / "test.csv",
)

# ------------------------------------------------------------
# Test : utiliser le vrai test.csv
# ------------------------------------------------------------

shutil.copy2(
    SOURCE_ANNOTATIONS / "train.csv",
    TEST_ANNOTATIONS_DIR / "train.csv",
)

shutil.copy2(
    SOURCE_ANNOTATIONS / "val.csv",
    TEST_ANNOTATIONS_DIR / "val.csv",
)

shutil.copy2(
    SOURCE_ANNOTATIONS / "test.csv",
    TEST_ANNOTATIONS_DIR / "test.csv",
)

# ------------------------------------------------------------
# Compter les vidéos
# ------------------------------------------------------------

validation_df = pd.read_csv(
    VAL_ANNOTATIONS_DIR / "test.csv",
    header=None,
)

test_df = pd.read_csv(
    TEST_ANNOTATIONS_DIR / "test.csv",
    header=None,
)

assert len(validation_df) == 335
assert len(test_df) == 335

print("=" * 70)
print("ÉVALUATION FINALE VIDEOMAE PRÉPARÉE")
print("=" * 70)

print("Checkpoint best :", BEST_CHECKPOINT)
print("Validation      :", len(validation_df), "vidéos")
print("Test            :", len(test_df), "vidéos")

print("\nDossier validation :", VAL_OUTPUT_DIR)
print("Dossier test       :", TEST_OUTPUT_DIR)

print(
    "\nChaque split produira : "
    "335 × 2 segments × 3 crops = 2010 vues"
)

print("\nPRÉPARATION TERMINÉE")

In [ ]:
# ============================================================
# VIDEOMAE — CELLULE 12B
# Évaluer checkpoint-best sur les 335 vidéos de validation
# ============================================================

import datetime
import os
import shutil
import subprocess
from pathlib import Path

# Vérifications
for path in [
    Path(PY),
    REPO_DIR,
    RUN_FILE,
    VIDEO_ROOT,
    VAL_ANNOTATIONS_DIR / "train.csv",
    VAL_ANNOTATIONS_DIR / "val.csv",
    VAL_ANNOTATIONS_DIR / "test.csv",
    BEST_CHECKPOINT,
]:
    assert path.exists(), path

# Nettoyer uniquement l'ancien résultat de validation
if VAL_OUTPUT_DIR.exists():
    shutil.rmtree(VAL_OUTPUT_DIR)

VAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Vérifier qu'aucune autre inférence ne tourne
process_check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps aux | grep run_class_finetuning.py "
        "| grep -v grep || true",
    ],
    text=True,
    capture_output=True,
)

if process_check.stdout.strip():
    print(process_check.stdout)
    raise RuntimeError(
        "Un processus VideoMAE est déjà actif."
    )

command_validation = [
    PY,
    "-m",
    "torch.distributed.run",
    "--nproc_per_node=2",
    "--master_port=29523",
    "run_class_finetuning.py",

    # Modèle
    "--model", "vit_base_patch16_224",
    "--nb_classes", "12",
    "--use_mean_pooling",
    "--init_scale", "0.001",

    # Données
    "--data_set", "RHAD",
    "--data_path", str(VAL_ANNOTATIONS_DIR),
    "--data_root", str(VIDEO_ROOT),

    # Configuration vidéo identique à l'entraînement
    "--input_size", "224",
    "--short_side_size", "224",
    "--num_frames", "16",
    "--num_segments", "1",
    "--tubelet_size", "2",
    "--sampling_rate", "4",

    # Évaluation
    "--batch_size", "2",
    "--num_workers", "2",
    "--seed", "42",
    "--test_num_segment", "2",
    "--test_num_crop", "3",

    # Charger uniquement les poids du meilleur checkpoint
    "--finetune", str(BEST_CHECKPOINT),
    "--model_key", "model",
    "--no_auto_resume",
    "--eval",

    # Sorties
    "--output_dir", str(VAL_OUTPUT_DIR),
    "--log_dir", str(VAL_OUTPUT_DIR),
]

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["OMP_NUM_THREADS"] = "1"
environment["PYTHONUNBUFFERED"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"

timestamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

validation_log = (
    FINAL_EVAL_DIR
    / f"validation_console_{timestamp}.log"
)

print("=" * 70)
print("ÉVALUATION VIDEOMAE — VALIDATION")
print("=" * 70)
print("Checkpoint :", BEST_CHECKPOINT)
print("Vidéos     : 335")
print("Vues       : 2010")
print("Log        :", validation_log)
print("=" * 70 + "\n")

with validation_log.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        command_validation,
        cwd=str(REPO_DIR),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        "L'évaluation validation s'est arrêtée "
        f"avec le code {return_code}."
    )

raw_files = sorted(
    VAL_OUTPUT_DIR.glob("*.txt")
)

print("\nFichiers bruts créés :")

for path in raw_files:
    number_of_lines = len(
        path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()
    )

    print(
        path.name,
        "→",
        number_of_lines,
        "lignes",
    )

assert (VAL_OUTPUT_DIR / "0.txt").exists()
assert (VAL_OUTPUT_DIR / "1.txt").exists()

print("\nPRÉDICTIONS DE VALIDATION TERMINÉES")

In [ ]:
# ============================================================
# VIDEOMAE — CELLULE 12C
# Évaluer checkpoint-best sur les 335 vidéos de test
# ============================================================

import datetime
import os
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

for path in [
    Path(PY),
    REPO_DIR,
    RUN_FILE,
    VIDEO_ROOT,
    TEST_ANNOTATIONS_DIR / "train.csv",
    TEST_ANNOTATIONS_DIR / "val.csv",
    TEST_ANNOTATIONS_DIR / "test.csv",
    BEST_CHECKPOINT,
]:
    assert path.exists(), path

# ------------------------------------------------------------
# Nettoyer uniquement une ancienne évaluation test
# ------------------------------------------------------------

if TEST_OUTPUT_DIR.exists():
    shutil.rmtree(TEST_OUTPUT_DIR)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Vérifier qu’aucun processus VideoMAE ne tourne
# ------------------------------------------------------------

process_check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps aux | grep run_class_finetuning.py "
        "| grep -v grep || true",
    ],
    text=True,
    capture_output=True,
)

if process_check.stdout.strip():
    print(process_check.stdout)

    raise RuntimeError(
        "Un processus VideoMAE est déjà actif."
    )

# ------------------------------------------------------------
# Commande d’évaluation
# ------------------------------------------------------------

command_test = [
    PY,
    "-m",
    "torch.distributed.run",
    "--nproc_per_node=2",
    "--master_port=29524",
    "run_class_finetuning.py",

    # Modèle
    "--model", "vit_base_patch16_224",
    "--nb_classes", "12",
    "--use_mean_pooling",
    "--init_scale", "0.001",

    # Données
    "--data_set", "RHAD",
    "--data_path", str(TEST_ANNOTATIONS_DIR),
    "--data_root", str(VIDEO_ROOT),

    # Configuration vidéo
    "--input_size", "224",
    "--short_side_size", "224",
    "--num_frames", "16",
    "--num_segments", "1",
    "--tubelet_size", "2",
    "--sampling_rate", "4",

    # Évaluation
    "--batch_size", "2",
    "--num_workers", "2",
    "--seed", "42",
    "--test_num_segment", "2",
    "--test_num_crop", "3",

    # Charger uniquement les poids du meilleur checkpoint
    "--finetune", str(BEST_CHECKPOINT),
    "--model_key", "model",
    "--no_auto_resume",
    "--eval",

    # Sorties
    "--output_dir", str(TEST_OUTPUT_DIR),
    "--log_dir", str(TEST_OUTPUT_DIR),
]

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["OMP_NUM_THREADS"] = "1"
environment["PYTHONUNBUFFERED"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"

timestamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

test_log = (
    FINAL_EVAL_DIR
    / f"test_console_{timestamp}.log"
)

print("=" * 70)
print("ÉVALUATION VIDEOMAE — TEST")
print("=" * 70)
print("Checkpoint :", BEST_CHECKPOINT)
print("Vidéos     : 335")
print("Vues       : 2010")
print("Log        :", test_log)
print("=" * 70 + "\n")

with test_log.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        command_test,
        cwd=str(REPO_DIR),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        "L’évaluation test s’est arrêtée "
        f"avec le code {return_code}."
    )

# ------------------------------------------------------------
# Vérifier les sorties
# ------------------------------------------------------------

raw_files = sorted(
    TEST_OUTPUT_DIR.glob("*.txt")
)

print("\nFichiers bruts créés :")

for path in raw_files:
    number_of_lines = len(
        path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()
    )

    print(
        path.name,
        "→",
        number_of_lines,
        "lignes",
    )

assert (TEST_OUTPUT_DIR / "0.txt").exists()
assert (TEST_OUTPUT_DIR / "1.txt").exists()

print("\n PRÉDICTIONS DE TEST TERMINÉES")

In [ ]:
# ============================================================
# CORRIGER LE FORMAT DES ANNOTATIONS VALIDATION ET TEST
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


def repair_annotation_file(path: Path):
    assert path.exists(), path

    original_lines = path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    parsed_rows = []

    for line_number, line in enumerate(
        original_lines,
        start=1,
    ):
        line = line.strip()

        if not line:
            continue

        # Accepte :
        # chemin,label
        # ou
        # chemin label
        if "," in line:
            video_path, label_text = line.rsplit(",", 1)
        else:
            try:
                video_path, label_text = line.rsplit(
                    maxsplit=1
                )
            except ValueError as error:
                raise ValueError(
                    f"Ligne {line_number} illisible dans {path} : {line}"
                ) from error

        video_path = video_path.strip()
        label_text = label_text.strip()

        try:
            label = int(label_text)
        except ValueError as error:
            raise ValueError(
                f"Label invalide à la ligne {line_number} : "
                f"{label_text!r}"
            ) from error

        assert video_path
        assert 0 <= label < 12

        parsed_rows.append(
            {
                "video_path": video_path,
                "annotation_label": label,
            }
        )

    repaired_df = pd.DataFrame(parsed_rows)

    assert len(repaired_df) == 335, (
        f"{path.name} contient {len(repaired_df)} lignes "
        "au lieu de 335."
    )

    assert repaired_df[
        "annotation_label"
    ].notna().all()

    assert repaired_df[
        "annotation_label"
    ].between(0, 11).all()

    # Sauvegarder l’original avant modification
    backup_path = path.with_suffix(
        path.suffix + ".before_format_fix"
    )

    if not backup_path.exists():
        shutil.copy2(path, backup_path)

    # Réécrire avec une vraie virgule
    repaired_df.to_csv(
        path,
        index=False,
        header=False,
    )

    print("=" * 70)
    print("Fichier corrigé :", path)
    print("Nombre de vidéos :", len(repaired_df))
    print("Labels manquants :", repaired_df["annotation_label"].isna().sum())
    print("\nPremières lignes :")
    print(repaired_df.head())


repair_annotation_file(
    VAL_ANNOTATION_FILE
)

repair_annotation_file(
    TEST_ANNOTATION_FILE
)

print("\nANNOTATIONS CORRIGÉES")
print("Tu peux relancer uniquement la cellule 12D.")

In [ ]:
# ============================================================
# VIDEOMAE — CELLULE 12D
# Fusionner les 6 vues et créer les CSV validation/test
# ============================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# ============================================================
# CONFIGURATION
# ============================================================

FINAL_EVAL_DIR = Path(
    "/kaggle/working/videomae_final_evaluation"
)

VAL_RAW_DIR = FINAL_EVAL_DIR / "validation_raw"
TEST_RAW_DIR = FINAL_EVAL_DIR / "test_raw"

VAL_ANNOTATION_FILE = (
    FINAL_EVAL_DIR
    / "annotations_validation"
    / "test.csv"
)

TEST_ANNOTATION_FILE = (
    FINAL_EVAL_DIR
    / "annotations_test"
    / "test.csv"
)

CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

NUM_CLASSES = len(CLASS_NAMES)
NORMAL_CLASS_INDEX = CLASS_NAMES.index("normal")

assert NUM_CLASSES == 12
assert NORMAL_CLASS_INDEX == 7


# ============================================================
# OUTILS
# ============================================================

def normalize_video_id(value):
    """
    Transforme un chemin ou un identifiant en nom de vidéo
    sans extension.
    """
    return Path(str(value).strip()).stem


def stable_softmax(logits):
    logits = np.asarray(
        logits,
        dtype=np.float64,
    )

    logits = logits - np.max(logits)
    exponentials = np.exp(logits)

    return exponentials / exponentials.sum()


def read_raw_predictions(raw_directory):
    """
    Lit 0.txt et 1.txt.

    Format d'une ligne :
    video_id [logit_0, ..., logit_11] label segment crop
    """

    raw_files = [
        raw_directory / "0.txt",
        raw_directory / "1.txt",
    ]

    rows = []

    for raw_file in raw_files:
        assert raw_file.exists(), raw_file

        lines = raw_file.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()

        for line in lines:

            # Ignore la ligne d'en-tête :
            # 100.0, 100.0
            if "[" not in line or "]" not in line:
                continue

            video_part, remaining = line.split(
                "[",
                1,
            )

            logits_text, metadata_text = remaining.split(
                "]",
                1,
            )

            video_id = normalize_video_id(
                video_part
            )

            logits = np.fromstring(
                logits_text,
                dtype=np.float64,
                sep=",",
            )

            metadata = metadata_text.strip().split()

            assert len(logits) == NUM_CLASSES, (
                f"{len(logits)} logits trouvés "
                f"pour {video_id}"
            )

            assert len(metadata) >= 3, line

            true_label = int(metadata[0])
            segment_index = int(metadata[1])
            crop_index = int(metadata[2])

            row = {
                "video_id": video_id,
                "true_label": true_label,
                "segment_index": segment_index,
                "crop_index": crop_index,
            }

            for class_index in range(NUM_CLASSES):
                row[f"logit_{class_index}"] = float(
                    logits[class_index]
                )

            rows.append(row)

    raw_df = pd.DataFrame(rows)

    assert len(raw_df) == 2010, (
        f"{len(raw_df)} vues trouvées au lieu de 2010"
    )

    print(
        raw_directory.name,
        "→",
        len(raw_df),
        "vues brutes",
    )

    return raw_df


def create_split_predictions(
    raw_directory,
    annotation_file,
    split_name,
):
    """
    Moyenne les logits des six vues de chaque vidéo,
    calcule les probabilités et produit le CSV final.
    """

    raw_df = read_raw_predictions(
        raw_directory
    )

    annotations_df = pd.read_csv(
        annotation_file,
        header=None,
        names=[
            "video_path",
            "annotation_label",
        ],
    )

    assert len(annotations_df) == 335

    annotations_df["video_id"] = (
        annotations_df["video_path"]
        .map(normalize_video_id)
    )

    assert (
        annotations_df["video_id"].nunique()
        == 335
    ), (
        "Les identifiants des vidéos dans les "
        "annotations ne sont pas uniques."
    )

    raw_video_ids = set(
        raw_df["video_id"]
    )

    annotation_video_ids = set(
        annotations_df["video_id"]
    )

    missing_in_raw = (
        annotation_video_ids - raw_video_ids
    )

    unexpected_in_raw = (
        raw_video_ids - annotation_video_ids
    )

    assert not missing_in_raw, (
        "Vidéos absentes des prédictions brutes : "
        f"{sorted(missing_in_raw)[:10]}"
    )

    assert not unexpected_in_raw, (
        "Vidéos inattendues dans les prédictions : "
        f"{sorted(unexpected_in_raw)[:10]}"
    )

    rows = []

    logit_columns = [
        f"logit_{class_index}"
        for class_index in range(NUM_CLASSES)
    ]

    # Utiliser l'ordre du fichier d'annotations.
    for _, annotation_row in annotations_df.iterrows():

        video_id = annotation_row["video_id"]
        video_path = str(
            annotation_row["video_path"]
        )

        annotation_label = int(
            annotation_row["annotation_label"]
        )

        video_views = raw_df[
            raw_df["video_id"] == video_id
        ]

        assert len(video_views) == 6, (
            f"{video_id} possède "
            f"{len(video_views)} vues au lieu de 6"
        )

        raw_labels = video_views[
            "true_label"
        ].unique()

        assert len(raw_labels) == 1
        assert int(raw_labels[0]) == annotation_label

        # Même méthode que la fusion VideoMAE :
        # moyenne des logits des six vues.
        mean_logits = (
            video_views[logit_columns]
            .to_numpy(dtype=np.float64)
            .mean(axis=0)
        )

        probabilities = stable_softmax(
            mean_logits
        )

        predicted_label = int(
            np.argmax(probabilities)
        )

        confidence = float(
            probabilities[predicted_label]
        )

        top5_labels = np.argsort(
            -probabilities
        )[:5]

        output_row = {
            "model": "VideoMAE",
            "split": split_name,
            "video_id": video_id,
            "video_path": video_path,
            "true_label": annotation_label,
            "true_class": CLASS_NAMES[
                annotation_label
            ],
            "predicted_label": predicted_label,
            "predicted_class": CLASS_NAMES[
                predicted_label
            ],
            "confidence": confidence,
            "correct_top1": int(
                predicted_label
                == annotation_label
            ),
            "correct_top5": int(
                annotation_label
                in top5_labels
            ),
            "view_count": int(
                len(video_views)
            ),
        }

        for class_index in range(NUM_CLASSES):
            output_row[
                f"prob_{class_index}"
            ] = float(
                probabilities[class_index]
            )

        rows.append(output_row)

    predictions_df = pd.DataFrame(rows)

    assert len(predictions_df) == 335
    assert predictions_df["video_id"].nunique() == 335
    assert predictions_df["view_count"].eq(6).all()
    assert predictions_df["confidence"].between(
        0,
        1,
    ).all()

    probability_columns = [
        f"prob_{class_index}"
        for class_index in range(NUM_CLASSES)
    ]

    probability_sums = (
        predictions_df[
            probability_columns
        ].sum(axis=1)
    )

    assert np.allclose(
        probability_sums,
        1.0,
        atol=1e-6,
    )

    # ========================================================
    # MÉTRIQUES
    # ========================================================

    true_labels = predictions_df[
        "true_label"
    ].to_numpy(dtype=int)

    predicted_labels = predictions_df[
        "predicted_label"
    ].to_numpy(dtype=int)

    top1_accuracy = accuracy_score(
        true_labels,
        predicted_labels,
    )

    top5_accuracy = float(
        predictions_df[
            "correct_top5"
        ].mean()
    )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0,
    )

    abnormal_predicted_as_normal = int(
        (
            (true_labels != NORMAL_CLASS_INDEX)
            & (
                predicted_labels
                == NORMAL_CLASS_INDEX
            )
        ).sum()
    )

    confusion = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
    )

    report_text = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    report_dictionary = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "model": "VideoMAE",
        "split": split_name,
        "number_of_videos": 335,
        "views_per_video": 6,
        "top1_accuracy": float(
            top1_accuracy
        ),
        "top5_accuracy": float(
            top5_accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "weighted_f1": float(
            weighted_f1
        ),
        "normal_class_index": int(
            NORMAL_CLASS_INDEX
        ),
        "abnormal_predicted_as_normal": (
            abnormal_predicted_as_normal
        ),
    }

    # ========================================================
    # SAUVEGARDE
    # ========================================================

    csv_path = (
        FINAL_EVAL_DIR
        / f"videomae_{split_name}_predictions.csv"
    )

    metrics_path = (
        FINAL_EVAL_DIR
        / f"videomae_{split_name}_metrics.json"
    )

    report_path = (
        FINAL_EVAL_DIR
        / (
            f"videomae_{split_name}_"
            "classification_report.txt"
        )
    )

    report_json_path = (
        FINAL_EVAL_DIR
        / (
            f"videomae_{split_name}_"
            "classification_report.json"
        )
    )

    confusion_path = (
        FINAL_EVAL_DIR
        / (
            f"videomae_{split_name}_"
            "confusion_matrix.csv"
        )
    )

    predictions_df.to_csv(
        csv_path,
        index=False,
    )

    metrics_path.write_text(
        json.dumps(
            metrics,
            indent=2,
        ),
        encoding="utf-8",
    )

    report_path.write_text(
        report_text,
        encoding="utf-8",
    )

    report_json_path.write_text(
        json.dumps(
            report_dictionary,
            indent=2,
        ),
        encoding="utf-8",
    )

    pd.DataFrame(
        confusion,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ).to_csv(
        confusion_path
    )

    print("\n" + "=" * 70)
    print(
        f"VIDEOMAE — {split_name.upper()}"
    )
    print("=" * 70)

    print(
        f"Top-1       : "
        f"{top1_accuracy * 100:.3f}%"
    )

    print(
        f"Top-5       : "
        f"{top5_accuracy * 100:.3f}%"
    )

    print(
        f"Macro F1    : {macro_f1:.4f}"
    )

    print(
        f"Weighted F1 : {weighted_f1:.4f}"
    )

    print(
        "Anormales prédites normal :",
        abnormal_predicted_as_normal,
    )

    print("CSV créé :", csv_path)

    return predictions_df, metrics


# ============================================================
# VALIDATION
# ============================================================

(
    videomae_validation_df,
    videomae_validation_metrics,
) = create_split_predictions(
    raw_directory=VAL_RAW_DIR,
    annotation_file=VAL_ANNOTATION_FILE,
    split_name="validation",
)


# ============================================================
# TEST
# ============================================================

(
    videomae_test_df,
    videomae_test_metrics,
) = create_split_predictions(
    raw_directory=TEST_RAW_DIR,
    annotation_file=TEST_ANNOTATION_FILE,
    split_name="test",
)


# ============================================================
# RÉSUMÉ ET ARCHIVE LÉGÈRE
# ============================================================

summary_df = pd.DataFrame(
    [
        videomae_validation_metrics,
        videomae_test_metrics,
    ]
)

summary_path = (
    FINAL_EVAL_DIR
    / "videomae_validation_test_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

archive_path = shutil.make_archive(
    base_name=(
        "/kaggle/working/"
        "videomae_predictions_and_metrics"
    ),
    format="zip",
    root_dir=str(FINAL_EVAL_DIR),
)

print("\n" + "=" * 70)
print("FICHIERS VIDEOMAE PRÊTS")
print("=" * 70)

print(
    FINAL_EVAL_DIR
    / "videomae_validation_predictions.csv"
)

print(
    FINAL_EVAL_DIR
    / "videomae_test_predictions.csv"
)

print("Archive :", archive_path)

print(
    "\nLES DEUX CSV DESTINÉS AU NOTEBOOK AT "
    "ONT ÉTÉ CRÉÉS"
)

In [ ]:
# ============================================================
# SAUVEGARDE FINALE VIDEOMAE DANS UN DATASET KAGGLE
# ============================================================

from pathlib import Path
import json
import os
import shutil
import kagglehub

TRAIN_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

EVAL_DIR = Path(
    "/kaggle/working/videomae_final_evaluation"
)

RESULTS_ZIP = Path(
    "/kaggle/working/videomae_predictions_and_metrics.zip"
)

UPLOAD_DIR = Path(
    "/kaggle/working/videomae_final_upload"
)

DATASET_HANDLE = (
    "nisrineamalkhnifass/videomae-final-rhad-results"
)

BEST_SOURCE = TRAIN_DIR / "checkpoint-best.pth"
LATEST_SOURCE = TRAIN_DIR / "checkpoint-latest.pth"

VAL_CSV = (
    EVAL_DIR / "videomae_validation_predictions.csv"
)

TEST_CSV = (
    EVAL_DIR / "videomae_test_predictions.csv"
)

for path in [
    BEST_SOURCE,
    LATEST_SOURCE,
    VAL_CSV,
    TEST_CSV,
    RESULTS_ZIP,
]:
    assert path.exists(), path

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def add_large_file(source: Path, destination: Path):
    """
    Crée un lien physique pour éviter de recopier les gros fichiers.
    """
    try:
        os.link(source, destination)
        print("Lien créé :", destination.name)
    except OSError:
        shutil.copy2(source, destination)
        print("Copie créée :", destination.name)


# Checkpoints
add_large_file(
    BEST_SOURCE,
    UPLOAD_DIR / "checkpoint-best.pth",
)

add_large_file(
    LATEST_SOURCE,
    UPLOAD_DIR / "checkpoint-latest.pth",
)

# CSV indispensables pour comparison_and_AT
shutil.copy2(
    VAL_CSV,
    UPLOAD_DIR / VAL_CSV.name,
)

shutil.copy2(
    TEST_CSV,
    UPLOAD_DIR / TEST_CSV.name,
)

# Archive avec métriques, rapports et matrices
shutil.copy2(
    RESULTS_ZIP,
    UPLOAD_DIR / RESULTS_ZIP.name,
)

# Logs d'entraînement
for pattern in [
    "*.log",
    "log.txt",
    "events.out.*",
]:
    for source in TRAIN_DIR.glob(pattern):
        if source.is_file():
            shutil.copy2(
                source,
                UPLOAD_DIR / source.name,
            )

# Résumé explicite
summary = {
    "model": "VideoMAE vit_base_patch16_224",
    "training_epochs": 40,
    "final_epoch": 39,
    "best_validation_accuracy_during_training_percent": 93.1343,
    "standardized_validation": {
        "videos": 335,
        "top1_percent": 91.343,
        "top5_percent": 99.104,
        "macro_f1": 0.8968,
        "weighted_f1": 0.9136,
        "abnormal_predicted_as_normal": 0,
    },
    "standardized_test": {
        "videos": 335,
        "top1_percent": 88.657,
        "top5_percent": 100.0,
        "macro_f1": 0.8151,
        "weighted_f1": 0.8818,
        "abnormal_predicted_as_normal": 0,
    },
    "normal_class_index": 7,
    "files_for_AT": [
        "videomae_validation_predictions.csv",
        "videomae_test_predictions.csv",
    ],
}

(UPLOAD_DIR / "videomae_final_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\nFichiers préparés :")

for path in sorted(UPLOAD_DIR.iterdir()):
    print(
        " -",
        path.name,
        f"({path.stat().st_size / 1024**2:.2f} MiB)",
    )

print("\n" + "=" * 70)
print("UPLOAD VIDEOMAE FINAL VERS KAGGLE")
print("Ne ferme pas la session avant la confirmation.")
print("=" * 70 + "\n")

kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIR),
    version_notes=(
        "VideoMAE final RHAD: checkpoints, logs, "
        "validation/test predictions and metrics."
    ),
)

print("\n" + "=" * 70)
print("SAUVEGARDE VIDEOMAE TERMINÉE")
print("=" * 70)

print("Dataset :", DATASET_HANDLE)
print(
    "Page :",
    f"https://www.kaggle.com/datasets/{DATASET_HANDLE}",
)

print(
    "\nVIDEOMAE EST ENTIÈREMENT SÉCURISÉ "
    "ET PRÊT POUR LE NOTEBOOK AT"
)

In [ ]:
from pathlib import Path
import torch

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

LATEST = OUTPUT_DIR / "checkpoint-latest.pth"
BEST = OUTPUT_DIR / "checkpoint-best.pth"

for path in [LATEST, BEST]:
    print("=" * 70)
    print("Fichier :", path)
    print("Existe  :", path.exists())

    if path.exists():
        print(
            "Taille  :",
            round(path.stat().st_size / 1024**2, 2),
            "MiB",
        )

assert LATEST.exists(), (
    "checkpoint-latest.pth est introuvable"
)

checkpoint = torch.load(
    LATEST,
    map_location="cpu",
    weights_only=False,
)

print("\nDernière epoch enregistrée :", checkpoint.get("epoch"))
print("L'ENTRAÎNEMENT PEUT ÊTRE REPRIS")

In [ ]:
from pathlib import Path
import json

SOURCE = Path(
    "/kaggle/working/videomae_rhad_clean_output/"
    "checkpoint-latest.pth"
)

BACKUP_DIR = Path(
    "/kaggle/working/VIDEOMAE_LATEST_PARTS"
)

BACKUP_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 250 * 1024 * 1024

assert SOURCE.exists(), SOURCE

parts = []

with SOURCE.open("rb") as source_file:
    index = 0

    while True:
        data = source_file.read(CHUNK_SIZE)

        if not data:
            break

        part_path = BACKUP_DIR / (
            f"checkpoint-latest.pth.part{index:03d}"
        )

        with part_path.open("wb") as part_file:
            part_file.write(data)

        parts.append(part_path.name)

        print(
            part_path.name,
            "→",
            round(part_path.stat().st_size / 1024**2, 2),
            "MiB",
        )

        index += 1

manifest = {
    "original_filename": SOURCE.name,
    "original_size_bytes": SOURCE.stat().st_size,
    "saved_epoch": 31,
    "parts": parts,
}

(BACKUP_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\nCheckpoint découpé")
print("Dossier :", BACKUP_DIR)

In [ ]:
# VIDEOMAE — CELLULE 12
# Vérifier les fichiers à télécharger avant de fermer Kaggle

from pathlib import Path
import json

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

files = [
    OUTPUT_DIR / "checkpoint-best.pth",
    OUTPUT_DIR / "checkpoint-latest.pth",
    OUTPUT_DIR / "log.txt",
]

print("=" * 70)
print("FICHIERS À TÉLÉCHARGER")
print("=" * 70)

for path in files:
    print(
        path,
        "->",
        path.exists(),
        (
            f"({path.stat().st_size / 1024**2:.2f} MiB)"
            if path.exists()
            else ""
        ),
    )

assert all(path.exists() for path in files)

records = []

for line in (
    OUTPUT_DIR / "log.txt"
).read_text(
    encoding="utf-8",
    errors="replace",
).splitlines():

    try:
        records.append(json.loads(line))
    except json.JSONDecodeError:
        pass

epochs = [
    record
    for record in records
    if (
        isinstance(record.get("epoch"), int)
        and isinstance(record.get("val_acc1"), (int, float))
    )
]

if epochs:
    latest = max(epochs, key=lambda record: record["epoch"])
    best = max(epochs, key=lambda record: record["val_acc1"])

    print("\nDernière epoch :", latest["epoch"])
    print("Meilleure epoch :", best["epoch"])
    print(
        "Meilleure Val Acc@1 :",
        f"{best['val_acc1']:.3f}%",
    )

print(
    "\nTélécharge checkpoint-best.pth, "
    "checkpoint-latest.pth et log.txt."
)


In [ ]:
# ============================================================
# SAUVEGARDER LE CHECKPOINT EPOCH 31 DANS UN DATASET KAGGLE
# ============================================================

from pathlib import Path
import json
import os
import shutil
import torch
import kagglehub

SOURCE = Path(
    "/kaggle/working/videomae_rhad_clean_output/"
    "checkpoint-latest.pth"
)

UPLOAD_DIR = Path(
    "/kaggle/working/videomae_resume_epoch31_upload"
)

TARGET = UPLOAD_DIR / "checkpoint-latest.pth"

DATASET_HANDLE = (
    "nisrineamalkhnifass/videomae-resume-epoch31"
)

# ------------------------------------------------------------
# 1. Vérifier le checkpoint
# ------------------------------------------------------------

assert SOURCE.exists(), f"Checkpoint introuvable : {SOURCE}"

checkpoint = torch.load(
    SOURCE,
    map_location="cpu",
    weights_only=False,
)

saved_epoch = checkpoint.get("epoch")

assert saved_epoch == 31, (
    f"Epoch trouvée : {saved_epoch}, au lieu de 31"
)

del checkpoint

print("Checkpoint vérifié")
print("Epoch :", saved_epoch)
print(
    "Taille :",
    round(SOURCE.stat().st_size / 1024**2, 2),
    "MiB",
)

# ------------------------------------------------------------
# 2. Créer un dossier contenant uniquement le fichier utile
# ------------------------------------------------------------

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Crée un lien physique : pas de seconde copie de 987 MiB.
try:
    os.link(SOURCE, TARGET)
    print("Lien physique créé sans recopier le fichier.")
except OSError:
    print("Lien impossible, copie normale en cours...")
    shutil.copy2(SOURCE, TARGET)

resume_information = {
    "model": "VideoMAE",
    "saved_epoch": int(saved_epoch),
    "next_epoch": 32,
    "original_path": str(SOURCE),
    "checkpoint_filename": TARGET.name,
    "size_bytes": TARGET.stat().st_size,
}

(UPLOAD_DIR / "resume_information.json").write_text(
    json.dumps(
        resume_information,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nDossier prêt :", UPLOAD_DIR)
print("Fichiers :", [p.name for p in UPLOAD_DIR.iterdir()])

# ------------------------------------------------------------
# 3. Envoyer directement vers un Dataset Kaggle
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UPLOAD VERS KAGGLE EN COURS")
print("Ne ferme pas la session avant la fin.")
print("=" * 70 + "\n")

result = kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIR),
    version_notes=(
        "VideoMAE checkpoint-latest sauvegardé "
        "à la fin de l'epoch 31. Reprise à l'epoch 32."
    ),
)

print("\n" + "=" * 70)
print("UPLOAD TERMINÉ")
print("=" * 70)
print("Dataset :", DATASET_HANDLE)
print("Résultat :", result)
print(
    "Page :",
    f"https://www.kaggle.com/datasets/{DATASET_HANDLE}",
)
print("\nLE CHECKPOINT EPOCH 31 EST MAINTENANT SÉCURISÉ")

In [ ]:
# ============================================================
# SAUVEGARDER CHECKPOINT-BEST + LOGS DANS UN DATASET KAGGLE
# ============================================================

from pathlib import Path
import json
import os
import shutil
import torch
import kagglehub

SOURCE_DIR = Path(
    "/kaggle/working/videomae_rhad_clean_output"
)

BEST_SOURCE = SOURCE_DIR / "checkpoint-best.pth"

UPLOAD_DIR = Path(
    "/kaggle/working/videomae_best_and_logs_upload"
)

DATASET_HANDLE = (
    "nisrineamalkhnifass/videomae-best-and-logs-epoch31"
)

assert BEST_SOURCE.exists(), (
    f"Checkpoint best introuvable : {BEST_SOURCE}"
)

# ------------------------------------------------------------
# Vérifier le checkpoint-best
# ------------------------------------------------------------

best_checkpoint = torch.load(
    BEST_SOURCE,
    map_location="cpu",
    weights_only=False,
)

best_saved_epoch = best_checkpoint.get("epoch")

print("Checkpoint-best vérifié")
print("Epoch contenue :", best_saved_epoch)
print(
    "Taille :",
    round(BEST_SOURCE.stat().st_size / 1024**2, 2),
    "MiB",
)

del best_checkpoint

# ------------------------------------------------------------
# Préparer le dossier d'upload
# ------------------------------------------------------------

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_TARGET = UPLOAD_DIR / "checkpoint-best.pth"

# Lien physique pour éviter de recopier 987 MiB
try:
    os.link(BEST_SOURCE, BEST_TARGET)
    print("Lien physique créé pour checkpoint-best.")
except OSError:
    print("Lien impossible, copie du checkpoint-best...")
    shutil.copy2(BEST_SOURCE, BEST_TARGET)

# ------------------------------------------------------------
# Copier tous les fichiers de logs disponibles
# ------------------------------------------------------------

log_candidates = []

for pattern in [
    "*.log",
    "*.txt",
    "*.json",
    "*.jsonl",
]:
    log_candidates.extend(
        SOURCE_DIR.glob(pattern)
    )

for log_path in sorted(set(log_candidates)):
    if log_path.is_file():
        shutil.copy2(
            log_path,
            UPLOAD_DIR / log_path.name,
        )
        print("Log ajouté :", log_path.name)

# ------------------------------------------------------------
# Informations de sauvegarde
# ------------------------------------------------------------

backup_information = {
    "model": "VideoMAE",
    "checkpoint_best_epoch": best_saved_epoch,
    "latest_checkpoint_saved_epoch": 31,
    "resume_from_epoch": 32,
    "best_checkpoint_filename": "checkpoint-best.pth",
    "best_checkpoint_size_bytes": BEST_TARGET.stat().st_size,
}

(UPLOAD_DIR / "backup_information.json").write_text(
    json.dumps(
        backup_information,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nFichiers qui seront envoyés :")

for path in sorted(UPLOAD_DIR.iterdir()):
    print(
        " -",
        path.name,
        f"({path.stat().st_size / 1024**2:.2f} MiB)",
    )

# ------------------------------------------------------------
# Upload vers Kaggle
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UPLOAD DU CHECKPOINT-BEST ET DES LOGS")
print("Ne ferme pas la session avant la confirmation.")
print("=" * 70 + "\n")

kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIR),
    version_notes=(
        "VideoMAE checkpoint-best et logs sauvegardés "
        "après interruption à l'epoch 31."
    ),
)

print("\n" + "=" * 70)
print("UPLOAD TERMINÉ")
print("=" * 70)
print("Dataset :", DATASET_HANDLE)
print(
    "Page :",
    f"https://www.kaggle.com/datasets/{DATASET_HANDLE}",
)
print("\nCHECKPOINT-BEST ET LOGS SÉCURISÉS")